
# MandiMitra: Agricultural Price Forecasting

## 1. Objective

Develop and evaluate short-term forecasting models for selected agricultural commodity-market price series in India.

The objective is to investigate whether historical modal prices can provide useful short-term price estimates and identify the commodity-market series for which forecasts are sufficiently reliable.

## 2. Scope

- Use the 74 forecasting series selected through the time-series suitability analysis in Notebook 03.
- Analyse historical modal prices at the commodity-market-variety-grade level.
- Establish naive and seasonal-naive forecasting baselines.
- Develop machine-learning forecasting models using historical price patterns.
- Evaluate predictions using chronological validation rather than random train-test splits.
- Compare model performance across different commodities and markets.
- Export validated forecasting results for integration into the MandiMitra application.

## 3. Methodology

The forecasting workflow will include:

1. Dataset validation and time-series inspection.
2. Chronological training, validation and testing.
3. Baseline forecasting.
4. Leakage-safe feature engineering.
5. Machine-learning model development.
6. Forecast accuracy evaluation using MAE, RMSE and sMAPE.
7. Error analysis and forecast reliability assessment.

## 4. Important Considerations

- Forecasts represent estimates, not guaranteed future market prices.
- Missing observations and irregular reporting intervals must be handled carefully.
- Future information must never leak into training features.
- Forecast performance must be compared against simple baselines.
- Forecast reliability may differ substantially across commodities and markets.


In [10]:

from pathlib import Path

import duckdb
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Project paths
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

PROCESSED_DIR = ROOT / "data" / "processed"
OUTPUT_DIR = PROCESSED_DIR / "forecasts"

# Input datasets from Notebook 03
CANDIDATE_FILE = PROCESSED_DIR / "forecasting_candidates.parquet"
PRICE_FILE = PROCESSED_DIR / "forecasting_candidate_prices.parquet"

# Create forecast output directory
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Display settings
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

sns.set_theme(style="whitegrid")

print("Project root:", ROOT)
print("Processed data directory:", PROCESSED_DIR)
print("Forecast output directory:", OUTPUT_DIR)

print("\nInput files:")
print("Candidate metadata exists:", CANDIDATE_FILE.exists())
print("Candidate prices exist:", PRICE_FILE.exists())


Project root: c:\Users\Lenovo\Documents\GrittingCode\MandiMitra
Processed data directory: c:\Users\Lenovo\Documents\GrittingCode\MandiMitra\data\processed
Forecast output directory: c:\Users\Lenovo\Documents\GrittingCode\MandiMitra\data\processed\forecasts

Input files:
Candidate metadata exists: True
Candidate prices exist: True


In [11]:

con = duckdb.connect()

for file_path in [CANDIDATE_FILE, PRICE_FILE]:

    print("\n" + "=" * 75)
    print(file_path.name)
    print("=" * 75)

    # Dataset schema
    schema = con.execute(
        "DESCRIBE SELECT * FROM read_parquet(?)",
        [str(file_path)]
    ).df()

    print("\nSchema:")
    display(schema)

    # Total records
    row_count = con.execute(
        "SELECT COUNT(*) FROM read_parquet(?)",
        [str(file_path)]
    ).fetchone()[0]

    print(f"\nTotal records: {row_count:,}")

    # Sample records
    sample = con.execute(
        "SELECT * FROM read_parquet(?) LIMIT 5",
        [str(file_path)]
    ).df()

    print("\nSample records:")
    display(sample)



forecasting_candidates.parquet

Schema:


,column_name,column_type,null,key,default,extra
0,State,VARCHAR,YES,None,None,None
1,District,VARCHAR,YES,None,None,None
2,Market,VARCHAR,YES,None,None,None
3,Project_Commodity,VARCHAR,YES,None,None,None
4,Variety,VARCHAR,YES,None,None,None
5,Grade,VARCHAR,YES,None,None,None
6,Observation_Count,BIGINT,YES,None,None,None
7,First_Date,TIMESTAMP,YES,None,None,None
8,Last_Date,TIMESTAMP,YES,None,None,None
9,Reporting_Days,BIGINT,YES,None,None,None



Total records: 74

Sample records:


,State,District,Market,Project_Commodity,Variety,Grade,Observation_Count,First_Date,Last_Date,Reporting_Days,Median_Modal_Price,Calendar_Days,Calendar_Coverage_Pct,Observation_Category,Gap_Intervals,Median_Gap_Days,P90_Gap_Days,Maximum_Gap_Days,Gaps_Above_7_Days,Gaps_Above_14_Days,Gaps_Above_30_Days,Days_Before_Dataset_End,Recency_Category,Total_Intervals,Median_Gap_Days_Gap,P90_Gap_Days_Gap,Maximum_Gap_Days_Gap,Gaps_Above_30_Days_Gap,Pct_Gaps_Above_30_Days,Short_Interval_Changes,Median_Absolute_Pct_Change,P90_Absolute_Pct_Change,Pass_Observation_Count,Pass_Recency,Pass_Reporting_Gaps,Pass_Long_Gap_Rate,Pass_Price_Changes,Criteria_Passed,History_Years,Observations_Per_Year,Segment_ID,Recent_Segment_Start,Recent_Segment_End,Recent_Segment_Observations,Recent_Segment_Years,Recent_Observations_Per_Year,Candidate_Rank,Market_Identity_Key,Market_Rank
0,Kerala,Kozhikode (Calicut),Palayam,Amaranthus,Other,FAQ,4076,2003-01-27,2025-11-05,4076,"1,600.00",8319,49.00,Above 1000,"4,075.00",1.00,3.00,889.00,19.00,8.00,6.00,166,91-180 days,4075,1.00,3.00,889,6,0.15,"3,913.00",0.00,17.65,True,True,True,True,True,5,22.77,178.98,7.00,2010-05-14,2025-11-05,3934,15.48,254.14,1,Kerala_Kozhikode (Calicut)_Palayam,1
1,Kerala,Pathanamthitta,Parakkodu,Amaranthus,Other,FAQ,3361,2005-11-24,2025-11-05,3361,"2,800.00",7287,46.12,Above 1000,"3,360.00",1.00,4.00,256.00,36.00,8.00,3.00,166,91-180 days,3360,1.00,4.00,256,3,0.09,"2,741.00",7.89,20.00,True,True,True,True,True,5,19.95,168.49,4.00,2010-11-04,2025-11-05,3028,15.00,201.82,2,Kerala_Pathanamthitta_Parakkodu,2
2,Nagaland,Dimapur,Dimapur,Beans,Other,FAQ,2551,2014-07-08,2025-10-29,2551,"4,700.00",4132,61.74,Above 1000,"2,550.00",1.00,2.00,135.00,26.00,16.00,8.00,173,91-180 days,2550,1.00,2.00,135,8,0.31,"2,480.00",4.00,11.11,True,True,True,True,True,5,11.31,225.55,9.00,2017-03-09,2025-10-29,2410,8.64,278.91,1,Nagaland_Dimapur_Dimapur,1
3,Gujarat,Rajkot,Jasdan,Beans,Other,FAQ,2985,2007-01-10,2025-11-05,2985,"4,500.00",6875,43.42,Above 1000,"2,984.00",1.00,3.00,317.00,107.00,41.00,14.00,166,91-180 days,2984,1.00,3.00,317,14,0.47,"2,705.00",7.50,27.86,True,True,True,True,True,5,18.82,158.61,15.00,2021-05-27,2025-11-05,1032,4.44,232.25,2,Gujarat_Rajkot_Jasdan,2
4,Kerala,Kozhikode (Calicut),Palayam,Beetroot,Other,FAQ,4062,2003-01-27,2025-11-05,4062,"3,000.00",8319,48.83,Above 1000,"4,061.00",1.00,3.00,928.00,34.00,16.00,11.00,166,91-180 days,4061,1.00,3.00,928,11,0.27,"3,876.00",4.55,21.05,True,True,True,True,True,5,22.77,178.37,12.00,2010-05-14,2025-11-05,3907,15.48,252.39,1,Kerala_Kozhikode (Calicut)_Palayam,1



forecasting_candidate_prices.parquet

Schema:


,column_name,column_type,null,key,default,extra
0,State,VARCHAR,YES,None,None,None
1,District,VARCHAR,YES,None,None,None
2,Market,VARCHAR,YES,None,None,None
3,Project_Commodity,VARCHAR,YES,None,None,None
4,Variety,VARCHAR,YES,None,None,None
5,Grade,VARCHAR,YES,None,None,None
6,Parsed_Date,TIMESTAMP,YES,None,None,None
7,Modal_Price,DOUBLE,YES,None,None,None



Total records: 238,755

Sample records:


,State,District,Market,Project_Commodity,Variety,Grade,Parsed_Date,Modal_Price
0,Kerala,Kozhikode (Calicut),Palayam,Amaranthus,Other,FAQ,2010-05-14,"1,300.00"
1,Kerala,Kozhikode (Calicut),Palayam,Amaranthus,Other,FAQ,2010-05-17,"1,000.00"
2,Kerala,Kozhikode (Calicut),Palayam,Amaranthus,Other,FAQ,2010-05-21,"1,200.00"
3,Kerala,Kozhikode (Calicut),Palayam,Amaranthus,Other,FAQ,2010-05-24,"1,100.00"
4,Kerala,Kozhikode (Calicut),Palayam,Amaranthus,Other,FAQ,2010-05-28,"1,100.00"


In [12]:

validation_query = """
WITH candidates AS (
    SELECT *
    FROM read_parquet(?)
),
prices AS (
    SELECT *
    FROM read_parquet(?)
),
matched AS (
    SELECT
        p.*,
        c.Candidate_Rank,
        c.Recent_Segment_Start,
        c.Recent_Segment_End
    FROM prices p
    INNER JOIN candidates c
        ON p.State = c.State
        AND p.District = c.District
        AND p.Market = c.Market
        AND p.Project_Commodity = c.Project_Commodity
        AND p.Variety = c.Variety
        AND p.Grade = c.Grade
),
duplicate_groups AS (
    SELECT
        State, District, Market,
        Project_Commodity, Variety, Grade,
        Parsed_Date,
        COUNT(*) AS n
    FROM prices
    GROUP BY ALL
    HAVING COUNT(*) > 1
)
SELECT
    (SELECT COUNT(*) FROM candidates) AS Selected_Series,
    (SELECT COUNT(DISTINCT Candidate_Rank) FROM matched)
        AS Matched_Series,
    (SELECT COUNT(*) FROM prices) AS Total_Observations,
    (SELECT COUNT(DISTINCT Parsed_Date) FROM prices)
        AS Distinct_Observation_Dates,
    (SELECT MIN(Parsed_Date) FROM prices) AS First_Date,
    (SELECT MAX(Parsed_Date) FROM prices) AS Last_Date,
    (SELECT COUNT(*) FROM duplicate_groups)
        AS Duplicate_Series_Date_Groups,
    COUNT(*) FILTER (
        WHERE Modal_Price IS NULL
           OR NOT isfinite(Modal_Price)
           OR Modal_Price <= 0
    ) AS Invalid_Prices,
    COUNT(*) FILTER (
        WHERE Parsed_Date >= Recent_Segment_Start
          AND Parsed_Date <= Recent_Segment_End
    ) AS Observations_Inside_Recent_Segment,
    COUNT(*) FILTER (
        WHERE Parsed_Date < Recent_Segment_Start
           OR Parsed_Date > Recent_Segment_End
    ) AS Observations_Outside_Recent_Segment
FROM matched
"""

validation_summary = con.execute(
    validation_query,
    [str(CANDIDATE_FILE), str(PRICE_FILE)]
).df()

display(validation_summary)


,Selected_Series,Matched_Series,Total_Observations,Distinct_Observation_Dates,First_Date,Last_Date,Duplicate_Series_Date_Groups,Invalid_Prices,Observations_Inside_Recent_Segment,Observations_Outside_Recent_Segment
0,74,9,238755,8167,2002-04-19,2025-11-05,0,0,238755,0


In [13]:

series_validation = con.execute("""
WITH candidates AS (
    SELECT *
    FROM read_parquet(?)
),
prices AS (
    SELECT *
    FROM read_parquet(?)
),
matched AS (
    SELECT
        c.State,
        c.District,
        c.Market,
        c.Project_Commodity,
        c.Variety,
        c.Grade,
        c.Candidate_Rank,
        c.Recent_Segment_Start,
        c.Recent_Segment_End,
        COUNT(p.Parsed_Date) AS Price_Observations,
        MIN(p.Parsed_Date) AS First_Observed_Date,
        MAX(p.Parsed_Date) AS Last_Observed_Date
    FROM candidates c
    LEFT JOIN prices p
        ON p.State = c.State
        AND p.District = c.District
        AND p.Market = c.Market
        AND p.Project_Commodity = c.Project_Commodity
        AND p.Variety = c.Variety
        AND p.Grade = c.Grade
    GROUP BY ALL
)
SELECT
    COUNT(*) AS Selected_Series,
    COUNT(*) FILTER (WHERE Price_Observations > 0)
        AS Matched_Series,
    COUNT(*) FILTER (WHERE Price_Observations = 0)
        AS Unmatched_Series,
    MIN(Price_Observations) AS Minimum_Observations,
    MEDIAN(Price_Observations) AS Median_Observations,
    MAX(Price_Observations) AS Maximum_Observations,
    MIN(First_Observed_Date) AS Earliest_Observation,
    MAX(Last_Observed_Date) AS Latest_Observation
FROM matched
""", [str(CANDIDATE_FILE), str(PRICE_FILE)]).df()

display(series_validation)


,Selected_Series,Matched_Series,Unmatched_Series,Minimum_Observations,Median_Observations,Maximum_Observations,Earliest_Observation,Latest_Observation
0,74,74,0,538,"3,402.50",6051,2002-04-19,2025-11-05


In [14]:
series_profile_query = """
WITH candidates AS (
    SELECT *
    FROM read_parquet(?)
),
prices AS (
    SELECT *
    FROM read_parquet(?)
),
ordered_prices AS (
    SELECT
        *,
        date_diff(
            'day',
            LAG(Parsed_Date) OVER (
                PARTITION BY
                    State, District, Market,
                    Project_Commodity, Variety, Grade
                ORDER BY Parsed_Date
            ),
            Parsed_Date
        ) AS Gap_Days
    FROM prices
),
price_profile AS (
    SELECT
        State, District, Market,
        Project_Commodity, Variety, Grade,
        COUNT(*) AS Observations,
        MIN(Parsed_Date) AS First_Date,
        MAX(Parsed_Date) AS Last_Date,
        MEDIAN(Gap_Days) AS Median_Gap_Days,
        QUANTILE_CONT(Gap_Days, 0.90) AS P90_Gap_Days,
        MAX(Gap_Days) AS Maximum_Gap_Days
    FROM ordered_prices
    GROUP BY ALL
)
SELECT
    c.Candidate_Rank,
    c.State,
    c.Market,
    c.Project_Commodity,
    c.Variety,
    c.Grade,
    p.Observations,
    p.First_Date,
    p.Last_Date,
    ROUND(
        date_diff('day', p.First_Date, p.Last_Date) / 365.25,
        2
    ) AS History_Years,
    ROUND(
        p.Observations /
        NULLIF(
            date_diff('day', p.First_Date, p.Last_Date) / 365.25,
            0
        ),
        1
    ) AS Observations_Per_Year,
    p.Median_Gap_Days,
    p.P90_Gap_Days,
    p.Maximum_Gap_Days
FROM candidates c
INNER JOIN price_profile p
    USING (
        State, District, Market,
        Project_Commodity, Variety, Grade
    )
ORDER BY c.Candidate_Rank
"""

series_profile = con.execute(
    series_profile_query,
    [str(CANDIDATE_FILE), str(PRICE_FILE)]
).df()

display(series_profile)

,Candidate_Rank,State,Market,Project_Commodity,Variety,Grade,Observations,First_Date,Last_Date,History_Years,Observations_Per_Year,Median_Gap_Days,P90_Gap_Days,Maximum_Gap_Days
0,1,Himachal Pradesh,Kullu,Potato,Other,FAQ,5789,2002-04-19,2025-11-03,23.54,245.90,1.00,3.00,27
1,1,Nagaland,Dimapur,Beans,Other,FAQ,2410,2017-03-09,2025-10-29,8.64,278.90,1.00,2.00,16
2,1,Himachal Pradesh,Bhuntar,Peas Wet,Other,FAQ,5593,2002-10-25,2025-10-29,23.01,243.10,1.00,3.00,29
3,1,Himachal Pradesh,Kullu,Capsicum,Other,FAQ,3787,2009-12-09,2025-11-01,15.90,238.20,1.00,3.00,29
4,1,Kerala,Palayam,Cowpea (Veg),Other,FAQ,3872,2010-05-14,2025-11-05,15.48,250.10,1.00,3.00,28
5,1,Gujarat,Vadhvan,Onion Green,Onion Green,FAQ,1613,2019-07-01,2025-11-01,6.34,254.50,1.00,2.00,18
6,1,Kerala,Palayam,Pumpkin,Other,FAQ,3862,2010-05-14,2025-11-05,15.48,249.50,1.00,3.00,24
7,1,Gujarat,Vadhvan,Cabbage,Cabbage,FAQ,5852,2005-01-15,2025-10-25,20.77,281.70,1.00,2.00,25
8,1,Kerala,Kondotty,Garlic,Other,FAQ,3673,2011-11-15,2025-11-05,13.97,262.80,1.00,2.00,27
9,1,Gujarat,Dahod (Veg. Market),Mint (Pudina),Mint (Pudina),FAQ,3517,2014-09-12,2025-11-04,11.15,315.50,1.00,2.00,27


In [15]:
gap_summary = (
    series_profile[
        [
            "Median_Gap_Days",
            "P90_Gap_Days",
            "Maximum_Gap_Days",
            "Observations_Per_Year"
        ]
    ]
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.90]
    )
    .T
)

display(gap_summary)

,count,mean,std,min,25%,50%,75%,90%,max
Median_Gap_Days,74.00,1.00,0.00,1.00,1.00,1.00,1.00,1.00,1.00
P90_Gap_Days,74.00,2.38,0.68,1.00,2.00,2.00,3.00,3.00,4.00
Maximum_Gap_Days,74.00,21.39,5.78,5.00,17.25,23.50,25.75,28.00,30.00
Observations_Per_Year,74.00,258.72,34.21,180.00,242.83,254.30,278.77,294.80,347.80


In [16]:
gap_summary = (
    series_profile[
        [
            "Median_Gap_Days",
            "P90_Gap_Days",
            "Maximum_Gap_Days",
            "Observations_Per_Year"
        ]
    ]
    .describe(
        percentiles=[0.25, 0.50, 0.75, 0.90]
    )
    .T
)

display(gap_summary)

,count,mean,std,min,25%,50%,75%,90%,max
Median_Gap_Days,74.00,1.00,0.00,1.00,1.00,1.00,1.00,1.00,1.00
P90_Gap_Days,74.00,2.38,0.68,1.00,2.00,2.00,3.00,3.00,4.00
Maximum_Gap_Days,74.00,21.39,5.78,5.00,17.25,23.50,25.75,28.00,30.00
Observations_Per_Year,74.00,258.72,34.21,180.00,242.83,254.30,278.77,294.80,347.80


In [17]:
calendar_split_query = """
SELECT
    State,
    District,
    Market,
    Project_Commodity,
    Variety,
    Grade,

    COUNT(*) AS Total_Observations,

    COUNT(*) FILTER (
        WHERE Parsed_Date < DATE '2024-01-01'
    ) AS Train_Observations,

    COUNT(*) FILTER (
        WHERE Parsed_Date >= DATE '2024-01-01'
          AND Parsed_Date < DATE '2025-01-01'
    ) AS Validation_Observations,

    COUNT(*) FILTER (
        WHERE Parsed_Date >= DATE '2025-01-01'
          AND Parsed_Date < DATE '2025-11-06'
    ) AS Test_Observations,

    MIN(Parsed_Date) AS First_Date,
    MAX(Parsed_Date) AS Last_Date

FROM read_parquet(?)

GROUP BY
    State, District, Market,
    Project_Commodity, Variety, Grade

ORDER BY Train_Observations
"""

global_split_detail = con.execute(
    calendar_split_query,
    [str(PRICE_FILE)]
).df()


# Minimum observations required in each partition.
MIN_TRAIN = 100
MIN_VALIDATION = 50
MIN_TEST = 50

global_usable = (
    (global_split_detail["Train_Observations"] >= MIN_TRAIN)
    & (global_split_detail["Validation_Observations"] >= MIN_VALIDATION)
    & (global_split_detail["Test_Observations"] >= MIN_TEST)
)

split_summary = pd.DataFrame({
    "Metric": [
        "Total selected series",
        "Series meeting all minimums",
        "Series failing at least one minimum",
        "Minimum training observations",
        "Minimum validation observations",
        "Minimum test observations"
    ],
    "Value": [
        len(global_split_detail),
        int(global_usable.sum()),
        int((~global_usable).sum()),
        global_split_detail["Train_Observations"].min(),
        global_split_detail["Validation_Observations"].min(),
        global_split_detail["Test_Observations"].min()
    ]
})

display(split_summary)


print("Series with insufficient observations:")

display(
    global_split_detail.loc[
        ~global_usable,
        [
            "State",
            "Market",
            "Project_Commodity",
            "Train_Observations",
            "Validation_Observations",
            "Test_Observations",
            "First_Date",
            "Last_Date"
        ]
    ]
)

,Metric,Value
0,Total selected series,74
1,Series meeting all minimums,74
2,Series failing at least one minimum,0
3,Minimum training observations,108
4,Minimum validation observations,167
5,Minimum test observations,117


Series with insufficient observations:


,State,Market,Project_Commodity,Train_Observations,Validation_Observations,Test_Observations,First_Date,Last_Date


In [18]:
split_query = """
SELECT
    State,
    District,
    Market,
    Project_Commodity,
    Variety,
    Grade,
    Parsed_Date,
    Modal_Price,
    CASE
        WHEN Parsed_Date < DATE '2024-01-01'
            THEN 'Train'
        WHEN Parsed_Date < DATE '2025-01-01'
            THEN 'Validation'
        ELSE 'Test'
    END AS Split
FROM read_parquet(?)
ORDER BY
    State,
    District,
    Market,
    Project_Commodity,
    Variety,
    Grade,
    Parsed_Date
"""

split_data = con.execute(
    split_query,
    [str(PRICE_FILE)]
).df()

split_summary = (
    split_data
    .groupby("Split")
    .agg(
        Observations=("Modal_Price", "size"),
        Series=("Market", "count"),
        First_Date=("Parsed_Date", "min"),
        Last_Date=("Parsed_Date", "max"),
        Average_Price=("Modal_Price", "mean")
    )
    .reindex(["Train", "Validation", "Test"])
)

display(split_summary)

,Observations,Series,First_Date,Last_Date,Average_Price
Split,,,,,
Train,202975,202975,2002-04-19,2023-12-31,"2,796.28"
Validation,19731,19731,2024-01-01,2024-12-31,"4,861.32"
Test,16049,16049,2025-01-01,2025-11-05,"4,169.32"


In [19]:
SERIES_KEYS = [
    "State",
    "District",
    "Market",
    "Project_Commodity",
    "Variety",
    "Grade"
]

observation_summary = (
    split_data
    .groupby("Split")
    .agg(
        Observations=("Modal_Price", "size"),
        First_Date=("Parsed_Date", "min"),
        Last_Date=("Parsed_Date", "max"),
        Average_Price=("Modal_Price", "mean")
    )
)

unique_series_summary = (
    split_data[["Split"] + SERIES_KEYS]
    .drop_duplicates()
    .groupby("Split")
    .size()
    .rename("Unique_Series")
)

split_summary = (
    observation_summary
    .join(unique_series_summary)
    .reindex(["Train", "Validation", "Test"])
)

display(split_summary)

,Observations,First_Date,Last_Date,Average_Price,Unique_Series
Split,,,,,
Train,202975,2002-04-19,2023-12-31,"2,796.28",74
Validation,19731,2024-01-01,2024-12-31,"4,861.32",74
Test,16049,2025-01-01,2025-11-05,"4,169.32",74


## 5. Forecasting Target and Baseline

### Forecasting Target

The target variable is the modal price of the next reported observation for each selected commodity-market-variety-grade series.

The forecasting task is one-step-ahead prediction based on previously reported observations.

### Baseline Model

The naive forecasting baseline predicts that the next reported modal price will equal the most recently observed modal price.

This baseline provides a reference against which more complex forecasting models will be evaluated.

### Evaluation Strategy

Models will be evaluated using chronological training, validation and test periods. No future observations will be used to construct historical features.

In [20]:
# Create next-observation target separately for each series

split_data = split_data.sort_values(
    SERIES_KEYS + ["Parsed_Date"]
).reset_index(drop=True)

split_data["Target_Next_Price"] = (
    split_data
    .groupby(SERIES_KEYS)["Modal_Price"]
    .shift(-1)
)

split_data["Target_Next_Date"] = (
    split_data
    .groupby(SERIES_KEYS)["Parsed_Date"]
    .shift(-1)
)

display(
    split_data[
        SERIES_KEYS
        + [
            "Parsed_Date",
            "Modal_Price",
            "Target_Next_Date",
            "Target_Next_Price",
            "Split"
        ]
    ].head(10)
)

,State,District,Market,Project_Commodity,Variety,Grade,Parsed_Date,Modal_Price,Target_Next_Date,Target_Next_Price,Split
0,Gujarat,Dahod,Dahod (Veg. Market),Drumstick,Other,FAQ,2014-09-12,"6,500.00",2014-09-13,"6,000.00",Train
1,Gujarat,Dahod,Dahod (Veg. Market),Drumstick,Other,FAQ,2014-09-13,"6,000.00",2014-09-14,"5,800.00",Train
2,Gujarat,Dahod,Dahod (Veg. Market),Drumstick,Other,FAQ,2014-09-14,"5,800.00",2014-09-15,"6,000.00",Train
3,Gujarat,Dahod,Dahod (Veg. Market),Drumstick,Other,FAQ,2014-09-15,"6,000.00",2014-09-16,"5,700.00",Train
4,Gujarat,Dahod,Dahod (Veg. Market),Drumstick,Other,FAQ,2014-09-16,"5,700.00",2014-09-17,"6,000.00",Train
5,Gujarat,Dahod,Dahod (Veg. Market),Drumstick,Other,FAQ,2014-09-17,"6,000.00",2014-09-18,"6,000.00",Train
6,Gujarat,Dahod,Dahod (Veg. Market),Drumstick,Other,FAQ,2014-09-18,"6,000.00",2014-09-19,"6,000.00",Train
7,Gujarat,Dahod,Dahod (Veg. Market),Drumstick,Other,FAQ,2014-09-19,"6,000.00",2014-09-20,"5,000.00",Train
8,Gujarat,Dahod,Dahod (Veg. Market),Drumstick,Other,FAQ,2014-09-20,"5,000.00",2014-09-21,"5,000.00",Train
9,Gujarat,Dahod,Dahod (Veg. Market),Drumstick,Other,FAQ,2014-09-21,"5,000.00",2014-09-22,"4,500.00",Train


In [21]:
# Assign each forecasting example according to its target date

split_data["Target_Split"] = np.select(
    [
        split_data["Target_Next_Date"] < pd.Timestamp("2024-01-01"),
        (
            (split_data["Target_Next_Date"] >= pd.Timestamp("2024-01-01"))
            & (split_data["Target_Next_Date"] < pd.Timestamp("2025-01-01"))
        ),
        (
            (split_data["Target_Next_Date"] >= pd.Timestamp("2025-01-01"))
            & (split_data["Target_Next_Date"] < pd.Timestamp("2025-11-06"))
        )
    ],
    ["Train", "Validation", "Test"],
    default="Outside"
)

# Exclude rows without a next reported observation
forecast_data = split_data.dropna(
    subset=["Target_Next_Price", "Target_Next_Date"]
).copy()

# Summarise the actual forecasting examples
target_split_summary = (
    forecast_data
    .groupby("Target_Split")
    .agg(
        Forecast_Examples=("Target_Next_Price", "size"),
        Unique_Series=("Market", "count"),
        First_Target_Date=("Target_Next_Date", "min"),
        Last_Target_Date=("Target_Next_Date", "max")
    )
    .reindex(["Train", "Validation", "Test"])
)

display(target_split_summary)

print("Target split distribution:")
display(
    forecast_data["Target_Split"]
    .value_counts(dropna=False)
    .rename("Rows")
    .to_frame()
)

,Forecast_Examples,Unique_Series,First_Target_Date,Last_Target_Date
Target_Split,,,,
Train,202901,202901,2002-04-22,2023-12-31
Validation,19731,19731,2024-01-01,2024-12-31
Test,16049,16049,2025-01-01,2025-11-05


Target split distribution:


,Rows
Target_Split,
Train,202901
Validation,19731
Test,16049


In [22]:
# Count genuinely distinct series in each target split

unique_series_by_split = (
    forecast_data[
        ["Target_Split"] + SERIES_KEYS
    ]
    .drop_duplicates()
    .groupby("Target_Split")
    .size()
    .rename("Unique_Series")
)

display(unique_series_by_split)

# Check whether every series appears in every split
series_coverage = (
    forecast_data
    .groupby(SERIES_KEYS)["Target_Split"]
    .nunique()
)

print("Series appearing in all three splits:", (series_coverage == 3).sum())
print("Series missing at least one split:", (series_coverage < 3).sum())

Target_Split
Test          74
Train         74
Validation    74
Name: Unique_Series, dtype: int64

Series appearing in all three splits: 74
Series missing at least one split: 0


In [23]:
# Naive baseline: next price = current observed price

forecast_data["Naive_Prediction"] = forecast_data["Modal_Price"]

# Calculate absolute and squared errors
forecast_data["Naive_Absolute_Error"] = (
    forecast_data["Target_Next_Price"]
    - forecast_data["Naive_Prediction"]
).abs()

forecast_data["Naive_Squared_Error"] = (
    forecast_data["Target_Next_Price"]
    - forecast_data["Naive_Prediction"]
) ** 2

# Calculate percentage error safely
forecast_data["Naive_Absolute_Percentage_Error"] = (
    forecast_data["Naive_Absolute_Error"]
    / forecast_data["Target_Next_Price"].abs()
) * 100

# Summarise baseline performance by split
naive_baseline_summary = (
    forecast_data
    .groupby("Target_Split")
    .agg(
        MAE=("Naive_Absolute_Error", "mean"),
        RMSE=("Naive_Squared_Error", lambda x: np.sqrt(x.mean())),
        MAPE=("Naive_Absolute_Percentage_Error", "mean"),
        Forecast_Examples=("Naive_Absolute_Error", "count")
    )
    .reindex(["Train", "Validation", "Test"])
)

display(naive_baseline_summary)

,MAE,RMSE,MAPE,Forecast_Examples
Target_Split,,,,
Train,213.09,491.45,9.35,202901
Validation,342.09,887.40,8.18,19731
Test,300.11,707.87,8.14,16049


In [24]:
# Calculate symmetric absolute percentage error

denominator = (
    forecast_data["Target_Next_Price"].abs()
    + forecast_data["Naive_Prediction"].abs()
)

forecast_data["Naive_SMAPE"] = np.where(
    denominator > 0,
    200 * forecast_data["Naive_Absolute_Error"] / denominator,
    0
)

# Evaluate performance by commodity and split

commodity_baseline = (
    forecast_data
    .groupby(["Project_Commodity", "Target_Split"])
    .agg(
        MAE=("Naive_Absolute_Error", "mean"),
        RMSE=("Naive_Squared_Error", lambda x: np.sqrt(x.mean())),
        MAPE=("Naive_Absolute_Percentage_Error", "mean"),
        sMAPE=("Naive_SMAPE", "mean"),
        Forecast_Examples=("Naive_Absolute_Error", "count")
    )
    .reset_index()
)

# Focus on test performance
test_commodity_baseline = (
    commodity_baseline[
        commodity_baseline["Target_Split"] == "Test"
    ]
    .sort_values("sMAPE", ascending=False)
    .reset_index(drop=True)
)

display(test_commodity_baseline)

,Project_Commodity,Target_Split,MAE,RMSE,MAPE,sMAPE,Forecast_Examples
0,French Beans (Frasbean),Test,624.32,944.12,14.88,14.42,333
1,Drumstick,Test,676.42,"1,117.99",14.47,13.98,458
2,Spinach,Test,257.11,469.54,14.27,13.63,415
3,Mint (Pudina),Test,524.28,984.79,14.26,13.58,523
4,Pumpkin,Test,133.56,264.08,13.69,13.22,441
5,Cluster beans,Test,558.66,840.64,12.08,11.82,447
6,Onion Green,Test,246.28,451.65,12.14,11.74,430
7,Indian Beans (Seam),Test,741.63,"1,196.59",11.57,11.32,448
8,Tomato,Test,219.12,394.50,10.90,10.66,336
9,Surat Beans (Papadi),Test,765.91,"1,297.54",11.56,10.55,242


In [25]:
# Create leakage-safe historical features

feature_data = forecast_data.copy()

# Sort chronologically within each series
feature_data = feature_data.sort_values(
    SERIES_KEYS + ["Parsed_Date"]
).reset_index(drop=True)

grouped_prices = feature_data.groupby(
    SERIES_KEYS,
    sort=False
)["Modal_Price"]

# Previous observed prices
for lag in [1, 2, 3, 7]:
    feature_data[f"Lag_{lag}_Price"] = grouped_prices.shift(lag)

# Rolling averages, including the current known observation
for window in [3, 7]:
    feature_data[f"Rolling_Mean_{window}"] = (
        feature_data
        .groupby(SERIES_KEYS, sort=False)["Modal_Price"]
        .transform(
            lambda s: s.rolling(window, min_periods=1).mean()
        )
    )

# Rolling volatility
feature_data["Rolling_Std_7"] = (
    feature_data
    .groupby(SERIES_KEYS, sort=False)["Modal_Price"]
    .transform(
        lambda s: s.rolling(7, min_periods=2).std()
    )
)

# Recent price changes
feature_data["Price_Change_1"] = (
    feature_data["Modal_Price"] - feature_data["Lag_1_Price"]
)

feature_data["Price_Change_3"] = (
    feature_data["Modal_Price"] - feature_data["Lag_3_Price"]
)

feature_data["Price_Pct_Change_1"] = (
    feature_data["Price_Change_1"]
    / feature_data["Lag_1_Price"].replace(0, np.nan)
)

# Calendar features
feature_data["Month"] = feature_data["Parsed_Date"].dt.month
feature_data["Day_Of_Week"] = feature_data["Parsed_Date"].dt.dayofweek
feature_data["Day_Of_Year"] = feature_data["Parsed_Date"].dt.dayofyear

# Remove examples without the required seven-observation history
feature_data = feature_data.dropna(
    subset=["Lag_7_Price", "Rolling_Std_7"]
).copy()

print("Forecasting examples after feature engineering:", len(feature_data))
print("Unique series:", feature_data[SERIES_KEYS].drop_duplicates().shape[0])

print("\nMissing values in engineered features:")
display(
    feature_data[
        [
            "Lag_1_Price",
            "Lag_2_Price",
            "Lag_3_Price",
            "Lag_7_Price",
            "Rolling_Mean_3",
            "Rolling_Mean_7",
            "Rolling_Std_7",
            "Price_Change_1",
            "Price_Change_3",
            "Price_Pct_Change_1"
        ]
    ].isna().sum().to_frame("Missing_Values")
)

display(
    feature_data[
        SERIES_KEYS
        + [
            "Parsed_Date",
            "Modal_Price",
            "Lag_1_Price",
            "Lag_3_Price",
            "Rolling_Mean_3",
            "Rolling_Std_7",
            "Target_Next_Price",
            "Target_Split"
        ]
    ].head(10)
)

Forecasting examples after feature engineering: 238163
Unique series: 74

Missing values in engineered features:


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_9680\2158852458.py:54: Pandas4Warning: Series.dt.dayofweek is deprecated and will be removed in a future version. Use Series.dt.day_of_week instead.
  feature_data["Day_Of_Week"] = feature_data["Parsed_Date"].dt.dayofweek
C:\Users\Lenovo\AppData\Local\Temp\ipykernel_9680\2158852458.py:55: Pandas4Warning: Series.dt.dayofyear is deprecated and will be removed in a future version. Use Series.dt.day_of_year instead.
  feature_data["Day_Of_Year"] = feature_data["Parsed_Date"].dt.dayofyear


,Missing_Values
Lag_1_Price,0
Lag_2_Price,0
Lag_3_Price,0
Lag_7_Price,0
Rolling_Mean_3,0
Rolling_Mean_7,0
Rolling_Std_7,0
Price_Change_1,0
Price_Change_3,0
Price_Pct_Change_1,0


,State,District,Market,Project_Commodity,Variety,Grade,Parsed_Date,Modal_Price,Lag_1_Price,Lag_3_Price,Rolling_Mean_3,Rolling_Std_7,Target_Next_Price,Target_Split
7,Gujarat,Dahod,Dahod (Veg. Market),Drumstick,Other,FAQ,2014-09-19,"6,000.00","6,000.00","5,700.00","6,000.00",125.36,"5,000.00",Train
8,Gujarat,Dahod,Dahod (Veg. Market),Drumstick,Other,FAQ,2014-09-20,"5,000.00","6,000.00","6,000.00","5,666.67",367.10,"5,000.00",Train
9,Gujarat,Dahod,Dahod (Veg. Market),Drumstick,Other,FAQ,2014-09-21,"5,000.00","5,000.00","6,000.00","5,333.33",471.57,"4,500.00",Train
10,Gujarat,Dahod,Dahod (Veg. Market),Drumstick,Other,FAQ,2014-09-22,"4,500.00","5,000.00","6,000.00","4,833.33",616.06,"4,800.00",Train
11,Gujarat,Dahod,Dahod (Veg. Market),Drumstick,Other,FAQ,2014-09-23,"4,800.00","4,500.00","5,000.00","4,766.67",649.91,"6,000.00",Train
12,Gujarat,Dahod,Dahod (Veg. Market),Drumstick,Other,FAQ,2014-09-25,"6,000.00","4,800.00","5,000.00","5,100.00",649.91,"6,000.00",Train
13,Gujarat,Dahod,Dahod (Veg. Market),Drumstick,Other,FAQ,2014-09-26,"6,000.00","6,000.00","4,500.00","5,600.00",649.91,"4,500.00",Train
14,Gujarat,Dahod,Dahod (Veg. Market),Drumstick,Other,FAQ,2014-09-30,"4,500.00","6,000.00","4,800.00","5,500.00",638.82,"6,000.00",Train
15,Gujarat,Dahod,Dahod (Veg. Market),Drumstick,Other,FAQ,2014-10-01,"6,000.00","4,500.00","6,000.00","5,500.00",716.14,"3,800.00",Train
16,Gujarat,Dahod,Dahod (Veg. Market),Drumstick,Other,FAQ,2014-10-02,"3,800.00","6,000.00","6,000.00","4,766.67",906.33,"4,300.00",Train


In [27]:
# Update deprecated date properties
feature_data["Day_Of_Week"] = feature_data["Parsed_Date"].dt.day_of_week
feature_data["Day_Of_Year"] = feature_data["Parsed_Date"].dt.day_of_year

# Define model features
FEATURE_COLUMNS = [
    "Modal_Price",
    "Lag_1_Price",
    "Lag_2_Price",
    "Lag_3_Price",
    "Lag_7_Price",
    "Rolling_Mean_3",
    "Rolling_Mean_7",
    "Rolling_Std_7",
    "Price_Change_1",
    "Price_Change_3",
    "Price_Pct_Change_1",
    "Month",
    "Day_Of_Week",
    "Day_Of_Year"
]

TARGET_COLUMN = "Target_Next_Price"

# Separate data chronologically
train_data = feature_data[
    feature_data["Target_Split"] == "Train"
].copy()

validation_data = feature_data[
    feature_data["Target_Split"] == "Validation"
].copy()

test_data = feature_data[
    feature_data["Target_Split"] == "Test"
].copy()

# Prepare model matrices
X_train = train_data[FEATURE_COLUMNS]
y_train = train_data[TARGET_COLUMN]

X_validation = validation_data[FEATURE_COLUMNS]
y_validation = validation_data[TARGET_COLUMN]

X_test = test_data[FEATURE_COLUMNS]
y_test = test_data[TARGET_COLUMN]

print("Training shape:", X_train.shape)
print("Validation shape:", X_validation.shape)
print("Test shape:", X_test.shape)

print("\nMissing values:")
print(X_train.isna().sum().sum())
print(X_validation.isna().sum().sum())
print(X_test.isna().sum().sum())

Training shape: (202383, 14)
Validation shape: (19731, 14)
Test shape: (16049, 14)

Missing values:
0
0
0


In [29]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Initialize model
rf_model = RandomForestRegressor(
    n_estimators=100,
    min_samples_leaf=5,
    max_features=0.8,
    random_state=42,
    n_jobs=-1
)

# Train using training data only
rf_model.fit(X_train, y_train)

# Predict validation prices
rf_validation_predictions = rf_model.predict(X_validation)

# Evaluation metrics
rf_mae = mean_absolute_error(
    y_validation, rf_validation_predictions
)

rf_rmse = np.sqrt(
    mean_squared_error(y_validation, rf_validation_predictions)
)

rf_smape = np.mean(
    200 * np.abs(y_validation - rf_validation_predictions) /
    (np.abs(y_validation) + np.abs(rf_validation_predictions) + 1e-8)
)

# Naive baseline on the exact same validation examples
naive_validation_predictions = validation_data["Modal_Price"]

naive_mae = mean_absolute_error(
    y_validation, naive_validation_predictions
)

naive_rmse = np.sqrt(
    mean_squared_error(y_validation, naive_validation_predictions)
)

naive_smape = np.mean(
    200 * np.abs(y_validation - naive_validation_predictions) /
    (np.abs(y_validation) + np.abs(naive_validation_predictions) + 1e-8)
)

# Compare models
validation_comparison = pd.DataFrame({
    "Model": ["Naive Baseline", "Random Forest"],
    "MAE": [naive_mae, rf_mae],
    "RMSE": [naive_rmse, rf_rmse],
    "sMAPE": [naive_smape, rf_smape]
})

validation_comparison

,Model,MAE,RMSE,sMAPE
0,Naive Baseline,342.09,887.40,7.83
1,Random Forest,391.36,880.01,8.67


In [30]:
validation_results = validation_data[
    ["Project_Commodity", "Target_Next_Price", "Modal_Price"]
].copy()

validation_results["Naive_Prediction"] = (
    validation_results["Modal_Price"]
)

validation_results["RF_Prediction"] = rf_validation_predictions

for model_name in ["Naive", "RF"]:
    prediction_col = f"{model_name}_Prediction"

    validation_results[f"{model_name}_Absolute_Error"] = (
        validation_results["Target_Next_Price"]
        - validation_results[prediction_col]
    ).abs()

    validation_results[f"{model_name}_sMAPE"] = (
        200
        * validation_results[f"{model_name}_Absolute_Error"]
        / (
            validation_results["Target_Next_Price"].abs()
            + validation_results[prediction_col].abs()
            + 1e-8
        )
    )

commodity_comparison = (
    validation_results
    .groupby("Project_Commodity")
    .agg(
        Examples=("Target_Next_Price", "count"),
        Naive_MAE=("Naive_Absolute_Error", "mean"),
        RF_MAE=("RF_Absolute_Error", "mean"),
        Naive_sMAPE=("Naive_sMAPE", "mean"),
        RF_sMAPE=("RF_sMAPE", "mean")
    )
)

commodity_comparison["sMAPE_Improvement_Pct"] = (
    (
        commodity_comparison["Naive_sMAPE"]
        - commodity_comparison["RF_sMAPE"]
    )
    / commodity_comparison["Naive_sMAPE"]
) * 100

commodity_comparison.sort_values(
    "sMAPE_Improvement_Pct",
    ascending=False
).round(2)

,Examples,Naive_MAE,RF_MAE,Naive_sMAPE,RF_sMAPE,sMAPE_Improvement_Pct
Project_Commodity,,,,,,
Onion,544,153.33,149.84,8.19,7.28,11.09
Cluster beans,537,544.88,516.22,12.79,12.13,5.15
French Beans (Frasbean),460,637.22,629.48,14.47,13.82,4.48
Beans,499,482.65,466.67,8.25,7.88,4.48
Mint (Pudina),622,488.62,483.13,12.22,11.99,1.88
Pumpkin,520,147.46,143.46,11.88,11.82,0.46
Drumstick,509,967.58,972.74,14.60,14.77,-1.16
Amaranthus,600,246.33,241.54,7.82,7.94,-1.57
Indian Beans (Seam),540,672.22,704.64,10.22,10.42,-1.94


In [31]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor

CATEGORICAL_COLUMNS = [
    "Project_Commodity",
    "State",
    "Market",
    "Variety",
    "Grade"
]

# Confirm all required columns exist
print("Missing categorical columns:",
      set(CATEGORICAL_COLUMNS) - set(feature_data.columns))

print("\nUnique category counts:")
print(feature_data[CATEGORICAL_COLUMNS].nunique())

# Prepare feature sets
X_train_full = train_data[
    FEATURE_COLUMNS + CATEGORICAL_COLUMNS
]

X_validation_full = validation_data[
    FEATURE_COLUMNS + CATEGORICAL_COLUMNS
]

print("\nTraining shape:", X_train_full.shape)
print("Validation shape:", X_validation_full.shape)

Missing categorical columns: set()

Unique category counts:
Project_Commodity    39
State                12
Market               37
Variety              20
Grade                 1
dtype: int64

Training shape: (202383, 19)
Validation shape: (19731, 19)


In [32]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

CATEGORICAL_COLUMNS = [
    "Project_Commodity",
    "State",
    "Market",
    "Variety"
]

ALL_FEATURES = FEATURE_COLUMNS + CATEGORICAL_COLUMNS

X_train_full = train_data[ALL_FEATURES]
X_validation_full = validation_data[ALL_FEATURES]

preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            CATEGORICAL_COLUMNS
        )
    ],
    remainder="passthrough"
)

rf_categorical_model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=100,
        min_samples_leaf=5,
        max_features=0.8,
        random_state=42,
        n_jobs=-1
    ))
])

rf_categorical_model.fit(X_train_full, y_train)

rf_cat_predictions = rf_categorical_model.predict(
    X_validation_full
)

rf_cat_mae = mean_absolute_error(
    y_validation, rf_cat_predictions
)

rf_cat_rmse = np.sqrt(
    mean_squared_error(y_validation, rf_cat_predictions)
)

rf_cat_smape = np.mean(
    200 * np.abs(y_validation - rf_cat_predictions) /
    (np.abs(y_validation) + np.abs(rf_cat_predictions) + 1e-8)
)

categorical_model_comparison = pd.DataFrame({
    "Model": [
        "Naive Baseline",
        "Random Forest",
        "Random Forest + Categories"
    ],
    "MAE": [
        naive_mae,
        rf_mae,
        rf_cat_mae
    ],
    "RMSE": [
        naive_rmse,
        rf_rmse,
        rf_cat_rmse
    ],
    "sMAPE": [
        naive_smape,
        rf_smape,
        rf_cat_smape
    ]
})

categorical_model_comparison.round(2)

,Model,MAE,RMSE,sMAPE
0,Naive Baseline,342.09,887.40,7.83
1,Random Forest,391.36,880.01,8.67
2,Random Forest + Categories,390.05,881.08,8.60


In [33]:
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

hgb_model = HistGradientBoostingRegressor(
    max_iter=200,
    learning_rate=0.05,
    max_leaf_nodes=31,
    l2_regularization=1.0,
    random_state=42
)

hgb_model.fit(X_train, y_train)

hgb_predictions = hgb_model.predict(X_validation)

hgb_mae = mean_absolute_error(
    y_validation, hgb_predictions
)

hgb_rmse = np.sqrt(
    mean_squared_error(y_validation, hgb_predictions)
)

hgb_smape = np.mean(
    200 * np.abs(y_validation - hgb_predictions) /
    (np.abs(y_validation) + np.abs(hgb_predictions) + 1e-8)
)

model_comparison = pd.DataFrame({
    "Model": [
        "Naive Baseline",
        "Random Forest",
        "Random Forest + Categories",
        "HistGradientBoosting"
    ],
    "MAE": [
        naive_mae, rf_mae, rf_cat_mae, hgb_mae
    ],
    "RMSE": [
        naive_rmse, rf_rmse, rf_cat_rmse, hgb_rmse
    ],
    "sMAPE": [
        naive_smape, rf_smape, rf_cat_smape, hgb_smape
    ]
})

model_comparison.round(2)

,Model,MAE,RMSE,sMAPE
0,Naive Baseline,342.09,887.40,7.83
1,Random Forest,391.36,880.01,8.67
2,Random Forest + Categories,390.05,881.08,8.60
3,HistGradientBoosting,452.37,"1,278.08",8.78


In [34]:
# Calculate log price movement
train_data["Target_Log_Change"] = np.log(
    train_data["Target_Next_Price"] / train_data["Modal_Price"]
)

validation_data["Target_Log_Change"] = np.log(
    validation_data["Target_Next_Price"] / validation_data["Modal_Price"]
)

# Inspect target distribution
print("Training target distribution:")
display(train_data["Target_Log_Change"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).to_frame("Log_Price_Change"))

print("\nValidation target distribution:")
display(validation_data["Target_Log_Change"].describe(
    percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99]
).to_frame("Log_Price_Change"))

print("\nInvalid target values:")
print(
    (~np.isfinite(train_data["Target_Log_Change"])).sum(),
    (~np.isfinite(validation_data["Target_Log_Change"])).sum()
)

Training target distribution:


,Log_Price_Change
count,"202,383.00"
mean,0.00
std,0.17
min,-3.26
1%,-0.51
5%,-0.25
25%,-0.04
50%,0.00
75%,0.04
95%,0.25



Validation target distribution:


,Log_Price_Change
count,"19,731.00"
mean,0.00
std,0.16
min,-1.61
1%,-0.51
5%,-0.22
25%,-0.02
50%,0.00
75%,0.02
95%,0.22



Invalid target values:
0 0


In [35]:
# Create log price change targets
train_data["Log_Price_Change"] = np.log(
    train_data["Target_Next_Price"] / train_data["Modal_Price"]
)

validation_data["Log_Price_Change"] = np.log(
    validation_data["Target_Next_Price"] / validation_data["Modal_Price"]
)

# Train model on log price changes
rf_log_model = RandomForestRegressor(
    n_estimators=100,
    min_samples_leaf=5,
    max_features=0.8,
    random_state=42,
    n_jobs=-1
)

rf_log_model.fit(
    X_train,
    train_data["Log_Price_Change"]
)

# Predict log price changes
predicted_log_changes = rf_log_model.predict(X_validation)

# Convert predictions back to price levels
rf_log_predictions = (
    validation_data["Modal_Price"].to_numpy()
    * np.exp(predicted_log_changes)
)

# Evaluate reconstructed prices
rf_log_mae = mean_absolute_error(
    y_validation, rf_log_predictions
)

rf_log_rmse = np.sqrt(
    mean_squared_error(y_validation, rf_log_predictions)
)

rf_log_smape = np.mean(
    200 * np.abs(y_validation - rf_log_predictions) /
    (np.abs(y_validation) + np.abs(rf_log_predictions) + 1e-8)
)

log_model_comparison = pd.DataFrame({
    "Model": [
        "Naive Baseline",
        "Random Forest - Price Level",
        "Random Forest - Log Change"
    ],
    "MAE": [
        naive_mae,
        rf_mae,
        rf_log_mae
    ],
    "RMSE": [
        naive_rmse,
        rf_rmse,
        rf_log_rmse
    ],
    "sMAPE": [
        naive_smape,
        rf_smape,
        rf_log_smape
    ]
})

log_model_comparison.round(2)

,Model,MAE,RMSE,sMAPE
0,Naive Baseline,342.09,887.40,7.83
1,Random Forest - Price Level,391.36,880.01,8.67
2,Random Forest - Log Change,383.81,845.79,8.60


In [36]:
log_validation_results = validation_data[
    ["Project_Commodity", "Target_Next_Price", "Modal_Price"]
].copy()

log_validation_results["Naive_Prediction"] = (
    log_validation_results["Modal_Price"]
)

log_validation_results["RF_Log_Prediction"] = rf_log_predictions

for model_name in ["Naive", "RF_Log"]:
    prediction_col = f"{model_name}_Prediction"

    log_validation_results[f"{model_name}_Absolute_Error"] = (
        log_validation_results["Target_Next_Price"]
        - log_validation_results[prediction_col]
    ).abs()

    log_validation_results[f"{model_name}_sMAPE"] = (
        200
        * log_validation_results[f"{model_name}_Absolute_Error"]
        / (
            log_validation_results["Target_Next_Price"].abs()
            + log_validation_results[prediction_col].abs()
            + 1e-8
        )
    )

log_commodity_comparison = (
    log_validation_results
    .groupby("Project_Commodity")
    .agg(
        Examples=("Target_Next_Price", "count"),
        Naive_MAE=("Naive_Absolute_Error", "mean"),
        RF_Log_MAE=("RF_Log_Absolute_Error", "mean"),
        Naive_sMAPE=("Naive_sMAPE", "mean"),
        RF_Log_sMAPE=("RF_Log_sMAPE", "mean")
    )
)

log_commodity_comparison["sMAPE_Improvement_Pct"] = (
    100
    * (
        log_commodity_comparison["Naive_sMAPE"]
        - log_commodity_comparison["RF_Log_sMAPE"]
    )
    / log_commodity_comparison["Naive_sMAPE"]
)

log_commodity_comparison.sort_values(
    "sMAPE_Improvement_Pct",
    ascending=False
).round(2)

,Examples,Naive_MAE,RF_Log_MAE,Naive_sMAPE,RF_Log_sMAPE,sMAPE_Improvement_Pct
Project_Commodity,,,,,,
Onion,544,153.33,153.15,8.19,7.53,8.03
Cluster beans,537,544.88,510.69,12.79,12.11,5.35
French Beans (Frasbean),460,637.22,622.61,14.47,13.91,3.88
Beans,499,482.65,476.87,8.25,8.02,2.85
Mint (Pudina),622,488.62,474.67,12.22,11.93,2.34
Pumpkin,520,147.46,142.12,11.88,11.65,1.88
Drumstick,509,967.58,970.23,14.60,14.61,-0.06
Amaranthus,600,246.33,244.98,7.82,7.99,-2.14
Indian Beans (Seam),540,672.22,706.78,10.22,10.45,-2.20


In [38]:
try:
    import xgboost as xgb
    print("XGBoost version:", xgb.__version__)
except ImportError:
    print("XGBoost is not installed.")

XGBoost version: 3.2.0


In [39]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=500,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=5,
    reg_lambda=5,
    objective="reg:squarederror",
    eval_metric="rmse",
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(
    X_train,
    train_data["Log_Price_Change"]
)

xgb_val_log_change = xgb_model.predict(X_validation)

xgb_val_predictions = (
    validation_data["Modal_Price"]
    * np.exp(xgb_val_log_change)
)

print("XGBoost training completed.")
print("Validation predictions:", len(xgb_val_predictions))
print("Sample predictions:")
print(xgb_val_predictions[:10])

XGBoost training completed.
Validation predictions: 19731
Sample predictions:
2663   7,134.53
2664   5,881.17
2665   5,622.30
2666   4,105.78
2667   4,994.54
2668   6,153.17
2669   5,453.24
2670   5,470.90
2671   4,717.10
2672   4,131.65
Name: Modal_Price, dtype: float64


In [40]:
xgb_errors = (
    validation_data["Target_Next_Price"].values
    - xgb_val_predictions.values
)

xgb_mae = np.mean(np.abs(xgb_errors))
xgb_rmse = np.sqrt(np.mean(xgb_errors ** 2))

xgb_smape = np.mean(
    200 * np.abs(xgb_errors) /
    (
        np.abs(validation_data["Target_Next_Price"].values)
        + np.abs(xgb_val_predictions.values)
        + 1e-8
    )
)

comparison = pd.DataFrame([
    {
        "Model": "Naive Baseline",
        "MAE": 342.09,
        "RMSE": 887.40,
        "sMAPE": 7.83
    },
    {
        "Model": "Random Forest (Price)",
        "MAE": 391.36,
        "RMSE": 880.01,
        "sMAPE": 8.67
    },
    {
        "Model": "Random Forest (Log Change)",
        "MAE": 383.81,
        "RMSE": 845.79,
        "sMAPE": 8.60
    },
    {
        "Model": "XGBoost (Log Change)",
        "MAE": xgb_mae,
        "RMSE": xgb_rmse,
        "sMAPE": xgb_smape
    }
])

comparison.sort_values("sMAPE")

,Model,MAE,RMSE,sMAPE
0,Naive Baseline,342.09,887.40,7.83
3,XGBoost (Log Change),365.74,826.27,8.27
2,Random Forest (Log Change),383.81,845.79,8.60
1,Random Forest (Price),391.36,880.01,8.67


In [41]:
xgb_commodity_data = validation_data[
    ["Project_Commodity", "Target_Next_Price", "Modal_Price"]
].copy()

xgb_commodity_data["XGBoost_Prediction"] = xgb_val_predictions.values
xgb_commodity_data["Naive_Prediction"] = (
    xgb_commodity_data["Modal_Price"]
)

def calculate_smape(actual, predicted):
    return np.mean(
        200 * np.abs(actual - predicted) /
        (np.abs(actual) + np.abs(predicted) + 1e-8)
    )

xgb_commodity_results = []

for commodity, group in xgb_commodity_data.groupby("Project_Commodity"):

    actual = group["Target_Next_Price"].values
    xgb_pred = group["XGBoost_Prediction"].values
    naive_pred = group["Naive_Prediction"].values

    naive_smape = calculate_smape(actual, naive_pred)
    xgb_smape = calculate_smape(actual, xgb_pred)

    xgb_commodity_results.append({
        "Commodity": commodity,
        "Observations": len(group),
        "Naive_sMAPE": naive_smape,
        "XGBoost_sMAPE": xgb_smape,
        "Improvement_Pct": (
            (naive_smape - xgb_smape) / naive_smape * 100
        )
    })

xgb_commodity_results = pd.DataFrame(xgb_commodity_results)

xgb_commodity_results = xgb_commodity_results.sort_values(
    "Improvement_Pct",
    ascending=False
)

print(
    "Commodities where XGBoost beats the naive baseline:",
    (xgb_commodity_results["Improvement_Pct"] > 0).sum(),
    "out of",
    len(xgb_commodity_results)
)

display(xgb_commodity_results)

Commodities where XGBoost beats the naive baseline: 10 out of 39


,Commodity,Observations,Naive_sMAPE,XGBoost_sMAPE,Improvement_Pct
25,Onion,544,8.19,7.38,9.88
11,Cluster beans,537,12.79,11.93,6.75
30,Pumpkin,520,11.88,11.39,4.14
17,French Beans (Frasbean),460,14.47,13.96,3.51
24,Mint (Pudina),622,12.22,11.80,3.43
1,Beans,499,8.25,7.98,3.26
0,Amaranthus,600,7.82,7.63,2.44
32,Surat Beans (Papadi),276,10.94,10.86,0.74
14,Drumstick,509,14.60,14.59,0.03
31,Spinach,516,11.92,11.92,0.00


In [47]:
actual = validation_data["Target_Next_Price"].values

predictions = {
    "Naive Baseline": validation_data["Modal_Price"].values,
    "Rolling Mean (3)": validation_data["Rolling_Mean_3"].values,
    "Rolling Mean (7)": validation_data["Rolling_Mean_7"].values,
    "Random Forest (Log Change)": np.asarray(rf_log_predictions),
    "XGBoost (Log Change)": np.asarray(xgb_val_predictions)
}

results = []

for model_name, predicted in predictions.items():

    errors = actual - predicted

    results.append({
        "Model": model_name,
        "MAE": np.mean(np.abs(errors)),
        "RMSE": np.sqrt(np.mean(errors ** 2)),
        "sMAPE": calculate_smape(actual, predicted)
    })

rolling_comparison = (
    pd.DataFrame(results)
    .sort_values("sMAPE")
    .reset_index(drop=True)
)

display(rolling_comparison)

,Model,MAE,RMSE,sMAPE
0,Naive Baseline,342.09,887.40,7.83
1,XGBoost (Log Change),365.74,826.27,8.27
2,Random Forest (Log Change),383.81,845.79,8.60
3,Rolling Mean (3),406.82,875.50,9.16
4,Rolling Mean (7),503.05,"1,002.55",11.20


In [48]:
commodity_training_counts = (
    train_data
    .groupby("Project_Commodity")
    .size()
    .sort_values()
    .reset_index(name="Training_Observations")
)

print("Number of commodities:", len(commodity_training_counts))
print("\nTraining observations per commodity:")
display(commodity_training_counts)

print("\nDistribution:")
display(
    commodity_training_counts["Training_Observations"]
    .describe(percentiles=[0.1, 0.25, 0.5, 0.75, 0.9])
)

Number of commodities: 39

Training observations per commodity:


,Project_Commodity,Training_Observations
0,Yam (Ratalu),703
1,Methi (Leaves),773
2,Surat Beans (Papadi),791
3,Duster Beans,1220
4,Onion Green,1961
5,Tinda,2023
6,Green Peas,2110
7,Cowpea (Lobia/Karamani),2121
8,Beans,2509
9,Sweet Potato,2745



Distribution:


count       39.00
mean     5,189.31
std      2,818.61
min        703.00
10%      1,812.80
25%      2,896.50
50%      5,369.00
75%      6,884.00
90%      8,714.80
max     10,791.00
Name: Training_Observations, dtype: float64

In [49]:
commodity_xgb_predictions = pd.Series(
    index=validation_data.index,
    dtype=float
)

commodity_xgb_models = {}

for commodity in train_data["Project_Commodity"].unique():

    commodity_train = train_data[
        train_data["Project_Commodity"] == commodity
    ]

    commodity_val = validation_data[
        validation_data["Project_Commodity"] == commodity
    ]

    if commodity_val.empty:
        continue

    model = XGBRegressor(
        n_estimators=300,
        max_depth=3,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=5,
        reg_lambda=5,
        objective="reg:squarederror",
        random_state=42,
        n_jobs=-1
    )

    model.fit(
        commodity_train[FEATURE_COLUMNS],
        commodity_train["Log_Price_Change"]
    )

    predicted_log_change = model.predict(
        commodity_val[FEATURE_COLUMNS]
    )

    commodity_xgb_predictions.loc[commodity_val.index] = (
        commodity_val["Modal_Price"].values
        * np.exp(predicted_log_change)
    )

    commodity_xgb_models[commodity] = model

print("Commodity-specific models trained:", len(commodity_xgb_models))
print("Validation predictions:", commodity_xgb_predictions.notna().sum())
print("Missing predictions:", commodity_xgb_predictions.isna().sum())

Commodity-specific models trained: 39
Validation predictions: 19731
Missing predictions: 0


In [50]:
actual = validation_data["Target_Next_Price"].values

predictions = {
    "Naive Baseline": validation_data["Modal_Price"].values,
    "Global XGBoost": np.asarray(xgb_val_predictions),
    "Commodity-Specific XGBoost": commodity_xgb_predictions.values
}

results = []

for model_name, predicted in predictions.items():

    errors = actual - predicted

    results.append({
        "Model": model_name,
        "MAE": np.mean(np.abs(errors)),
        "RMSE": np.sqrt(np.mean(errors ** 2)),
        "sMAPE": calculate_smape(actual, predicted)
    })

commodity_model_comparison = (
    pd.DataFrame(results)
    .sort_values("sMAPE")
    .reset_index(drop=True)
)

display(commodity_model_comparison)

,Model,MAE,RMSE,sMAPE
0,Naive Baseline,342.09,887.40,7.83
1,Global XGBoost,365.74,826.27,8.27
2,Commodity-Specific XGBoost,413.34,915.75,8.87


In [51]:
seasonal_check = (
    feature_data
    .groupby(SERIES_KEYS)
    .agg(
        First_Date=("Parsed_Date", "min"),
        Last_Date=("Parsed_Date", "max"),
        Observations=("Parsed_Date", "count")
    )
    .reset_index()
)

seasonal_check["History_Years"] = (
    seasonal_check["Last_Date"] - seasonal_check["First_Date"]
).dt.days / 365.25

print("Series with at least 2 years of engineered history:",
      (seasonal_check["History_Years"] >= 2).sum())

print("\nHistory duration:")
display(seasonal_check["History_Years"].describe())

print("\nSample:")
display(seasonal_check.head(10))

Series with at least 2 years of engineered history: 74

History duration:


count   74.00
mean    12.42
std      5.74
min      2.23
25%      7.76
50%     13.89
75%     15.34
max     23.49
Name: History_Years, dtype: float64


Sample:


,State,District,Market,Project_Commodity,Variety,Grade,First_Date,Last_Date,Observations,History_Years
0,Gujarat,Dahod,Dahod (Veg. Market),Drumstick,Other,FAQ,2014-09-19,2025-10-28,3173,11.11
1,Gujarat,Dahod,Dahod (Veg. Market),Mint (Pudina),Mint (Pudina),FAQ,2014-09-19,2025-10-23,3509,11.09
2,Gujarat,Rajkot,Jasdan,Beans,Other,FAQ,2021-06-14,2025-11-04,1024,4.39
3,Gujarat,Rajkot,Rajkot (Veg.Sub Yard),Cluster beans,Cluster Beans,FAQ,2016-09-02,2025-11-01,2485,9.16
4,Gujarat,Surat,Surat,Elephant Yam (Suran),Other,FAQ,2017-03-04,2025-11-03,2550,8.67
5,Gujarat,Surat,Surat,Pointed gourd (Parval),Other,FAQ,2017-03-04,2025-11-04,2526,8.67
6,Gujarat,Surat,Surat,Surat Beans (Papadi),Other,FAQ,2020-10-23,2025-11-03,1309,5.03
7,Gujarat,Surat,Surat,Sweet Potato,Other,FAQ,2020-11-21,2025-11-01,1314,4.94
8,Gujarat,Surat,Surat,Yam (Ratalu),Other,FAQ,2023-08-11,2025-11-04,612,2.23
9,Gujarat,Surendranagar,Vadhvan,Bottle gourd,Bottle Gourd,FAQ,2008-08-13,2025-10-28,4013,17.21


In [52]:
seasonal_predictions = pd.Series(
    np.nan,
    index=validation_data.index,
    dtype=float
)

historical_groups = {
    key: group.sort_values("Parsed_Date")
    for key, group in forecast_data.groupby(SERIES_KEYS)
}

for key, val_group in validation_data.groupby(SERIES_KEYS):

    history = historical_groups.get(key)

    if history is None:
        continue

    history_dates = history["Parsed_Date"].values
    history_prices = history["Modal_Price"].values

    for idx, row in val_group.iterrows():

        target_seasonal_date = (
            row["Parsed_Date"] - pd.Timedelta(days=365)
        )

        position = np.searchsorted(
            history_dates,
            np.datetime64(target_seasonal_date)
        )

        candidates = [
            p for p in [position - 1, position]
            if 0 <= p < len(history_dates)
        ]

        if not candidates:
            continue

        best_position = min(
            candidates,
            key=lambda p: abs(
                (pd.Timestamp(history_dates[p]) - target_seasonal_date).days
            )
        )

        matched_date = pd.Timestamp(history_dates[best_position])

        if (
            matched_date < row["Parsed_Date"]
            and abs((matched_date - target_seasonal_date).days) <= 7
        ):
            seasonal_predictions.loc[idx] = history_prices[best_position]

seasonal_coverage = seasonal_predictions.notna().sum()

print("Validation observations:", len(validation_data))
print("Seasonal predictions available:", seasonal_coverage)
print("Coverage:", f"{seasonal_coverage / len(validation_data) * 100:.2f}%")
print("Missing seasonal predictions:", seasonal_predictions.isna().sum())

Validation observations: 19731
Seasonal predictions available: 19469
Coverage: 98.67%
Missing seasonal predictions: 262


In [53]:
seasonal_mask = seasonal_predictions.notna()

actual = validation_data.loc[
    seasonal_mask, "Target_Next_Price"
].values

predictions = {
    "Naive Baseline": validation_data.loc[
        seasonal_mask, "Modal_Price"
    ].values,

    "Seasonal Naive (Previous Year)": seasonal_predictions.loc[
        seasonal_mask
    ].values,

    "Global XGBoost": xgb_val_predictions.loc[
        seasonal_mask
    ].values,

    "Random Forest (Log Change)": np.asarray(
        rf_log_predictions
    )[seasonal_mask.values],

    "Commodity-Specific XGBoost": commodity_xgb_predictions.loc[
        seasonal_mask
    ].values
}

results = []

for model_name, predicted in predictions.items():

    errors = actual - predicted

    results.append({
        "Model": model_name,
        "Observations": len(actual),
        "MAE": np.mean(np.abs(errors)),
        "RMSE": np.sqrt(np.mean(errors ** 2)),
        "sMAPE": calculate_smape(actual, predicted)
    })

seasonal_comparison = (
    pd.DataFrame(results)
    .sort_values("sMAPE")
    .reset_index(drop=True)
)

display(seasonal_comparison)

,Model,Observations,MAE,RMSE,sMAPE
0,Naive Baseline,19469,342.44,890.48,7.86
1,Global XGBoost,19469,366.17,829.40,8.31
2,Random Forest (Log Change),19469,384.24,848.98,8.64
3,Commodity-Specific XGBoost,19469,405.08,911.35,8.81
4,Seasonal Naive (Previous Year),19469,"1,692.55","3,043.63",35.56


In [54]:
evaluation_data = validation_data[
    SERIES_KEYS + ["Target_Next_Price", "Modal_Price"]
].copy()

evaluation_data["Naive"] = validation_data["Modal_Price"].values
evaluation_data["Global_XGBoost"] = np.asarray(xgb_val_predictions)
evaluation_data["Random_Forest"] = np.asarray(rf_log_predictions)
evaluation_data["Commodity_XGBoost"] = commodity_xgb_predictions.values

def smape_array(actual, predicted):
    return (
        200 * np.abs(actual - predicted)
        / (np.abs(actual) + np.abs(predicted) + 1e-8)
    )

for model in [
    "Naive",
    "Global_XGBoost",
    "Random_Forest",
    "Commodity_XGBoost"
]:
    evaluation_data[f"{model}_sMAPE"] = smape_array(
        evaluation_data["Target_Next_Price"].values,
        evaluation_data[model].values
    )

series_results = (
    evaluation_data
    .groupby(SERIES_KEYS)
    .agg({
        "Naive_sMAPE": "mean",
        "Global_XGBoost_sMAPE": "mean",
        "Random_Forest_sMAPE": "mean",
        "Commodity_XGBoost_sMAPE": "mean"
    })
)

macro_summary = pd.DataFrame({
    "Model": [
        "Naive Baseline",
        "Global XGBoost",
        "Random Forest",
        "Commodity-Specific XGBoost"
    ],
    "Mean_Series_sMAPE": [
        series_results["Naive_sMAPE"].mean(),
        series_results["Global_XGBoost_sMAPE"].mean(),
        series_results["Random_Forest_sMAPE"].mean(),
        series_results["Commodity_XGBoost_sMAPE"].mean()
    ],
    "Median_Series_sMAPE": [
        series_results["Naive_sMAPE"].median(),
        series_results["Global_XGBoost_sMAPE"].median(),
        series_results["Random_Forest_sMAPE"].median(),
        series_results["Commodity_XGBoost_sMAPE"].median()
    ]
}).sort_values("Mean_Series_sMAPE")

display(macro_summary)

print("\nSeries where each ML model beats the naive baseline:")

for model in ["Global_XGBoost", "Random_Forest", "Commodity_XGBoost"]:
    wins = (
        series_results[f"{model}_sMAPE"]
        < series_results["Naive_sMAPE"]
    ).sum()

    print(f"{model}: {wins} / {len(series_results)}")

,Model,Mean_Series_sMAPE,Median_Series_sMAPE
0,Naive Baseline,8.02,7.22
1,Global XGBoost,8.45,7.86
2,Random Forest,8.78,8.24
3,Commodity-Specific XGBoost,9.11,8.63



Series where each ML model beats the naive baseline:
Global_XGBoost: 22 / 74
Random_Forest: 16 / 74
Commodity_XGBoost: 16 / 74


In [55]:
# Isolate the untouched test observations
test_eval = feature_data[
    feature_data["Target_Split"] == "Test"
].copy()

# Generate predictions using the trained global XGBoost model
xgb_test_log_change = xgb_model.predict(
    test_eval[FEATURE_COLUMNS]
)

xgb_test_predictions = (
    test_eval["Modal_Price"].values
    * np.exp(xgb_test_log_change)
)

actual = test_eval["Target_Next_Price"].values

predictions = {
    "Naive Baseline": test_eval["Modal_Price"].values,
    "Global XGBoost": xgb_test_predictions
}

test_results = []

for model_name, predicted in predictions.items():

    errors = actual - predicted

    test_results.append({
        "Model": model_name,
        "Observations": len(actual),
        "MAE": np.mean(np.abs(errors)),
        "RMSE": np.sqrt(np.mean(errors ** 2)),
        "sMAPE": calculate_smape(actual, predicted)
    })

test_comparison = (
    pd.DataFrame(test_results)
    .sort_values("sMAPE")
    .reset_index(drop=True)
)

display(test_comparison)

,Model,Observations,MAE,RMSE,sMAPE
0,Naive Baseline,16049,300.11,707.87,7.83
1,Global XGBoost,16049,313.47,652.78,8.18


In [56]:
test_eval = test_eval.copy()

test_eval["Naive_Prediction"] = test_eval["Modal_Price"].values
test_eval["XGBoost_Prediction"] = xgb_test_predictions

actual = test_eval["Target_Next_Price"].values

test_eval["Naive_sMAPE"] = calculate_smape(
    actual, test_eval["Naive_Prediction"].values
)

test_eval["XGBoost_sMAPE"] = calculate_smape(
    actual, test_eval["XGBoost_Prediction"].values
)

test_series_results = (
    test_eval
    .groupby(SERIES_KEYS)
    .agg({
        "Naive_sMAPE": "mean",
        "XGBoost_sMAPE": "mean"
    })
)

test_macro_summary = pd.DataFrame({
    "Model": ["Naive Baseline", "Global XGBoost"],
    "Mean_Series_sMAPE": [
        test_series_results["Naive_sMAPE"].mean(),
        test_series_results["XGBoost_sMAPE"].mean()
    ],
    "Median_Series_sMAPE": [
        test_series_results["Naive_sMAPE"].median(),
        test_series_results["XGBoost_sMAPE"].median()
    ]
})

display(test_macro_summary)

print(
    "Series where XGBoost beats naive baseline:",
    (
        test_series_results["XGBoost_sMAPE"]
        < test_series_results["Naive_sMAPE"]
    ).sum(),
    "/",
    len(test_series_results)
)

,Model,Mean_Series_sMAPE,Median_Series_sMAPE
0,Naive Baseline,7.83,7.83
1,Global XGBoost,8.18,8.18


Series where XGBoost beats naive baseline: 0 / 74


## 10. Model Evaluation Conclusions

### Evaluation Setup

The forecasting experiment evaluated 74 selected commodity-market-variety-grade
series using chronological splits:

- Training: observations with target dates before 2024.
- Validation: observations with target dates in 2024.
- Testing: observations with target dates from January to November 2025.

The prediction target was the next reported modal price, not necessarily
the price exactly seven calendar days later.

### Model Comparison

The naive baseline predicted that the next reported price would equal
the latest observed modal price.

Global XGBoost, Random Forest, and commodity-specific XGBoost models
were evaluated as machine-learning alternatives.

The naive baseline achieved the best overall validation performance
and also outperformed Global XGBoost on the held-out test set.

On the test set, the naive baseline achieved:

- MAE: ₹300.11
- RMSE: ₹707.87
- sMAPE: 7.83%

Global XGBoost achieved:

- MAE: ₹313.47
- RMSE: ₹652.78
- sMAPE: 8.18%

The naive baseline had lower MAE and sMAPE. XGBoost had lower RMSE,
indicating better performance on squared errors despite its higher
average absolute and relative errors.

At the individual-series level, the naive baseline outperformed
Global XGBoost in all 74 test series.

### Conclusion

The naive baseline was selected as the primary forecasting approach
for the current experiment. More complex machine-learning models did
not demonstrate a consistent improvement over this simple benchmark.

This does not establish that the naive baseline is universally optimal.
The result applies to the selected 74 series and the next-reported-
observation forecasting task.

The original seven-calendar-day forecasting objective remains a
separate experiment and has not yet been validated.

All forecasts should be presented as estimates with clear uncertainty
and data-freshness information, not as guaranteed future prices.